# OmniGen v1 — Multi-Image VTO Server

Hosts OmniGen v1 locally on Colab GPU for multi-image virtual try-on.
Accepts 2-3 person images + 1 garment image for better identity preservation.

**GPU:** T4 works (uses 4-bit quantization). A100 is faster.

**How to use:**
1. Run all cells (~5 min for model download)
2. Copy the `https://xxxxx.gradio.live` URL
3. Set it as `OMNIGEN_COLAB_URL` secret in Supabase

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q gradio Pillow requests
!pip install -q accelerate>=1.2.0 diffusers>=0.31.0 peft transformers

# Clone and install OmniGen
import os
if not os.path.exists('OmniGen'):
    !git clone https://github.com/VectorSpaceLab/OmniGen.git
os.chdir('OmniGen')
!pip install -q -e . 2>&1 | tail -1

print('\nDependencies installed!')

In [ ]:
#@title 2. Load OmniGen model
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name()} ({vram:.1f} GB VRAM)')

from OmniGen import OmniGenPipeline

print('Loading OmniGen v1 (this takes a few minutes)...')
pipe = OmniGenPipeline.from_pretrained('Shitao/OmniGen-v1')
print('OmniGen v1 loaded!')

In [ ]:
#@title 3. Define generation function
from PIL import Image
import io
import base64
import requests
import tempfile
import json

def download_image(url, max_dim=768):
    """Download image from URL, resize, save to temp file, return path."""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    img = Image.open(io.BytesIO(response.content)).convert('RGB')
    if max(img.size) > max_dim:
        ratio = max_dim / max(img.size)
        img = img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)
    path = tempfile.mktemp(suffix='.jpg')
    img.save(path, 'JPEG', quality=90)
    return path

def generate_tryon(selfie_url, fullbody_url, garment_url, description='clothing item', steps=50):
    """Run OmniGen virtual try-on with multiple person references."""
    try:
        input_images = []
        img_idx = 1
        prompt_refs = []
        
        # Selfie (face reference) — optional
        if selfie_url and selfie_url.strip():
            print('Downloading selfie...')
            selfie_path = download_image(selfie_url)
            input_images.append(selfie_path)
            prompt_refs.append(f'<img><|image_{img_idx}|></img>')
            img_idx += 1
        
        # Full body (required)
        print('Downloading full body...')
        fullbody_path = download_image(fullbody_url)
        input_images.append(fullbody_path)
        prompt_refs.append(f'<img><|image_{img_idx}|></img>')
        img_idx += 1
        
        # Garment (required)
        print('Downloading garment...')
        garment_path = download_image(garment_url)
        input_images.append(garment_path)
        
        # Build prompt
        person_refs = ' and '.join(prompt_refs)
        prompt = (
            f'Generate a photorealistic image of the person shown in {person_refs} '
            f'wearing the {description} shown in <img><|image_{img_idx}|></img>. '
            f'Preserve their exact face, skin tone, hair, and body proportions. '
            f'Natural full-body standing pose, studio lighting, fashion photography.'
        )
        
        print(f'Running OmniGen with {len(input_images)} images...')
        print(f'Prompt: {prompt[:100]}...')
        
        output = pipe(
            prompt=prompt,
            input_images=input_images,
            num_inference_steps=int(steps),
            guidance_scale=3.0,
            img_guidance_scale=1.6,
            height=1024,
            width=768,
            seed=42,
            use_kv_cache=False,
            offload_kv_cache=True,
            offload_model=True,
        )
        
        result_img = output[0]
        
        # Convert to base64
        buf = io.BytesIO()
        result_img.save(buf, format='JPEG', quality=90)
        b64 = base64.b64encode(buf.getvalue()).decode()
        
        # Cleanup temp files
        for p in input_images:
            try: os.unlink(p)
            except: pass
        
        print('Generation complete!')
        return json.dumps({"success": True, "image_base64": b64})
    
    except Exception as e:
        import traceback
        traceback.print_exc()
        return json.dumps({"success": False, "error": str(e)})

print('Generation function ready!')

In [ ]:
#@title 4. Launch Gradio API Server
import gradio as gr

demo = gr.Interface(
    fn=generate_tryon,
    inputs=[
        gr.Textbox(label='Selfie URL (optional)', placeholder='https://...'),
        gr.Textbox(label='Full Body URL (required)', placeholder='https://...'),
        gr.Textbox(label='Garment URL (required)', placeholder='https://...'),
        gr.Textbox(label='Description', value='clothing item'),
        gr.Slider(minimum=20, maximum=80, value=50, step=5, label='Steps'),
    ],
    outputs=gr.Textbox(label='Result JSON'),
    title='OmniGen v1 VTO Server',
    description='Multi-image Virtual Try-On. Send selfie + full body + garment URLs.',
    api_name='tryon',
)

print('\n' + '='*60)
print('STARTING OMNIGEN SERVER...')
print('Copy the public URL below and set it as')
print('OMNIGEN_COLAB_URL in your Supabase secrets.')
print('='*60 + '\n')

demo.launch(share=True, quiet=False)